# Combining Spatial Enhancement Methods
**Bone Scan Image — Full Pipeline (a) → (h)**

| Step | Description |
|------|-------------|
| **(a)** | Original bone scan image |
| **(b)** | Laplacian of (a) — sharpens edges |
| **(c)** | (a) + (b) — basic sharpened image |
| **(d)** | Sobel gradient magnitude of (a) |
| **(e)** | Sobel image (d) smoothed with 5×5 box filter |
| **(f)** | Mask = (b) × (e) — product of Laplacian and smoothed Sobel |
| **(g)** | (a) + (f) — sharpened image using mask |
| **(h)** | Power-law (gamma) transformation applied to (g) — final result |

## 1. Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import convolve, uniform_filter
from skimage import io, img_as_float, filters
from skimage.color import rgb2gray
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## 2. Upload Bone Scan Image

In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]
image_raw = io.imread(filename)

print(f'Loaded : {filename}')
print(f'Shape  : {image_raw.shape}')
print(f'Dtype  : {image_raw.dtype}')

## 3. (a) Original Image — Convert to Grayscale Float

In [ ]:
if image_raw.ndim == 3:
    img_a = rgb2gray(img_as_float(image_raw))
else:
    img_a = img_as_float(image_raw)

print(f'Shape  : {img_a.shape}')
print(f'Range  : [{img_a.min():.4f}, {img_a.max():.4f}]')

plt.figure(figsize=(5, 8))
plt.imshow(img_a, cmap='gray')
plt.title('(a) Original Image', fontsize=13)
plt.axis('off'); plt.tight_layout(); plt.show()

## 4. (b) Laplacian Filter
Highlights edges and fine details by computing second-order derivatives.

In [ ]:
# Standard Laplacian kernel (with diagonal neighbors)
laplacian_kernel = np.array([
    [1,  1, 1],
    [1, -8, 1],
    [1,  1, 1]
], dtype=np.float64)

img_b = convolve(img_a, laplacian_kernel)

# Display scaled for visibility
img_b_display = (img_b - img_b.min()) / (img_b.max() - img_b.min())

plt.figure(figsize=(5, 8))
plt.imshow(img_b_display, cmap='gray')
plt.title('(b) Laplacian Image', fontsize=13)
plt.axis('off'); plt.tight_layout(); plt.show()

print(f'Laplacian range: [{img_b.min():.4f}, {img_b.max():.4f}]')

## 5. (c) Sharpened = (a) + (b)
Basic sharpening by adding original image and its Laplacian.

In [ ]:
img_c = img_a + img_b
# Clip to valid range
img_c = np.clip(img_c, 0, 1)

plt.figure(figsize=(5, 8))
plt.imshow(img_c, cmap='gray')
plt.title('(c) Sharpened: (a) + (b)', fontsize=13)
plt.axis('off'); plt.tight_layout(); plt.show()

## 6. (d) Sobel Gradient Magnitude
First-order derivative — highlights edges using Sobel operator.

In [ ]:
img_d = filters.sobel(img_a)

plt.figure(figsize=(5, 8))
plt.imshow(img_d, cmap='gray')
plt.title('(d) Sobel Gradient Magnitude', fontsize=13)
plt.axis('off'); plt.tight_layout(); plt.show()

print(f'Sobel range: [{img_d.min():.4f}, {img_d.max():.4f}]')

## 7. (e) Sobel Smoothed with 5×5 Box Filter
Smooth the Sobel image to reduce noise while preserving broad edge structure.

In [ ]:
# 5x5 averaging (box) filter
img_e = uniform_filter(img_d, size=5)

plt.figure(figsize=(5, 8))
plt.imshow(img_e, cmap='gray')
plt.title('(e) Sobel Smoothed (5×5 Box Filter)', fontsize=13)
plt.axis('off'); plt.tight_layout(); plt.show()

print(f'Smoothed Sobel range: [{img_e.min():.4f}, {img_e.max():.4f}]')

## 8. (f) Mask Image = (b) × (e)
Product of Laplacian and smoothed Sobel — combines sharpness info from both operators.

In [ ]:
img_f = img_b * img_e

# Normalize for display
img_f_display = (img_f - img_f.min()) / (img_f.max() - img_f.min())

plt.figure(figsize=(5, 8))
plt.imshow(img_f_display, cmap='gray')
plt.title('(f) Mask: (b) × (e)', fontsize=13)
plt.axis('off'); plt.tight_layout(); plt.show()

print(f'Mask range: [{img_f.min():.6f}, {img_f.max():.6f}]')

## 9. (g) Sharpened Image = (a) + (f)
Add mask to original to produce a sharper, detail-enhanced image.

In [ ]:
# Scale mask before adding so it has meaningful contribution
mask_scaled = (img_f - img_f.min()) / (img_f.max() - img_f.min())

img_g = img_a + mask_scaled
img_g = np.clip(img_g, 0, 1)

plt.figure(figsize=(5, 8))
plt.imshow(img_g, cmap='gray')
plt.title('(g) Sharpened: (a) + (f)', fontsize=13)
plt.axis('off'); plt.tight_layout(); plt.show()

## 10. (h) Final Result — Power-Law (Gamma) Transformation on (g)
$$s = c \cdot r^{\gamma}$$

- **gamma < 1** → brightens dark regions (bring out detail in shadows)
- **gamma > 1** → darkens bright regions

**Tune `gamma`** to best match the expected output.

In [ ]:
gamma = 0.4   # <-- adjust (try 0.3, 0.4, 0.5)
c = 1.0       # constant (usually 1.0)

img_h = c * np.power(img_g, gamma)
img_h = np.clip(img_h, 0, 1)

plt.figure(figsize=(5, 8))
plt.imshow(img_h, cmap='gray')
plt.title(f'(h) Power-Law on (g)  γ={gamma}', fontsize=13)
plt.axis('off'); plt.tight_layout(); plt.show()

print(f'Output range: [{img_h.min():.4f}, {img_h.max():.4f}]')

## 11. Full Pipeline — All Steps (a) to (h)

In [ ]:
images = [
    (img_a,         '(a) Original'),
    (img_b_display, '(b) Laplacian'),
    (img_c,         '(c) (a)+(b)'),
    (img_d,         '(d) Sobel'),
    (img_e,         '(e) Sobel + 5×5 Box'),
    (img_f_display, '(f) Mask (b)×(e)'),
    (img_g,         '(g) (a)+(f)'),
    (img_h,         f'(h) Power-Law γ={gamma}'),
]

fig, axes = plt.subplots(2, 4, figsize=(20, 12))
fig.suptitle('Combining Spatial Enhancement Methods — Full Pipeline',
             fontsize=15, fontweight='bold')

for ax, (img, title) in zip(axes.flatten(), images):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.savefig('result_full_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: result_full_pipeline.png')

## 12. Compare (a), (g), (h) — as the exam requires

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 8))
fig.suptitle('Compare: Original vs Sharpened vs Power-Law',
             fontsize=14, fontweight='bold')

axes[0].imshow(img_a, cmap='gray')
axes[0].set_title('(a) Original', fontsize=13)
axes[0].axis('off')

axes[1].imshow(img_g, cmap='gray')
axes[1].set_title('(g) Sharpened: (a)+(f)', fontsize=13)
axes[1].axis('off')

axes[2].imshow(img_h, cmap='gray')
axes[2].set_title(f'(h) Final: Power-Law γ={gamma}', fontsize=13)
axes[2].axis('off')

plt.tight_layout()
plt.savefig('result_comparison_a_g_h.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: result_comparison_a_g_h.png')

## 13. (Bonus) Try Different Gamma Values

In [ ]:
gammas = [0.2, 0.4, 0.6, 0.8, 1.0, 1.5]

fig, axes = plt.subplots(1, len(gammas), figsize=(22, 5))
fig.suptitle('Effect of Different Gamma Values on (g)', fontsize=13)

for ax, g in zip(axes, gammas):
    out = np.clip(np.power(img_g, g), 0, 1)
    ax.imshow(out, cmap='gray')
    ax.set_title(f'γ={g}', fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.savefig('result_gamma_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 14. Download Results

In [ ]:
files.download('result_full_pipeline.png')
files.download('result_comparison_a_g_h.png')
files.download('result_gamma_comparison.png')
print('Done! All results downloaded.')